In [ ]:
  # CELL A1: Imports
  import os
  import random
  import numpy as np
  import pandas as pd

  from sklearn.model_selection import train_test_split
  from sklearn.metrics import roc_auc_score, classification_report

  RANDOM_STATE = 42
  random.seed(RANDOM_STATE)
  np.random.seed(RANDOM_STATE)

  processed_dir = "../data/processed"
  clean_path = os.path.join(processed_dir, "clean_dataset.csv")

  df_clean = pd.read_csv(clean_path)
  df_clean = df_clean.sort_values(["pid", "position"]).reset_index(drop=True)

  print("Rows:", len(df_clean))
  print("Playlists:", df_clean["pid"].nunique())
  df_clean.head()

  # CELL A2: Build positive pairs from consecutive songs inside each playlist

  song_cols = [
      "track_id", "Track Name", "Artist Name(s)",
      "Danceability", "Energy", "Key", "Loudness", "Mode",
      "Speechiness", "Acousticness", "Instrumentalness",
      "Liveness", "Valence", "Tempo"
  ]

  context_cols = ["pid", "region_genre_id", "position", "pid_position"]

  next_df = df_clean.groupby("pid")[song_cols + context_cols].shift(-1)

  positive_pairs = pd.concat([
      df_clean[song_cols + context_cols].add_prefix("source_"),
      next_df.add_prefix("target_")
  ], axis=1)

  positive_pairs = positive_pairs.dropna(subset=["target_track_id"]).copy()
  positive_pairs["label"] = 1

  print("Positive pairs:", len(positive_pairs))
  positive_pairs.head()

  # CELL A3: Build negative pairs
  # Strategy: for each positive source song, sample random target songs that are NOT from the same playlist

  NEGATIVE_RATIO = 2  # 2 negatives per positive

  unique_tracks = (
      df_clean.sort_values(["track_id", "pid", "position"])
      .drop_duplicates(subset=["track_id"])
      .reset_index(drop=True)
  )

  track_feature_cols = [
      "track_id", "Track Name", "Artist Name(s)",
      "Danceability", "Energy", "Key", "Loudness", "Mode",
      "Speechiness", "Acousticness", "Instrumentalness",
      "Liveness", "Valence", "Tempo"
  ]

  track_lookup = unique_tracks[track_feature_cols].set_index("track_id").to_dict("index")
  all_track_ids = unique_tracks["track_id"].tolist()

  playlist_track_sets = df_clean.groupby("pid")["track_id"].apply(set).to_dict()

  negative_rows = []

  for row in positive_pairs.itertuples(index=False):
      source_pid = row.source_pid
      source_playlist_tracks = playlist_track_sets[source_pid]
      source_track_id = row.source_track_id

      valid_targets = [
          tid for tid in all_track_ids
          if tid not in source_playlist_tracks and tid != source_track_id
      ]

      sampled_targets = random.sample(valid_targets, k=min(NEGATIVE_RATIO, len(valid_targets)))

      for target_tid in sampled_targets:
          target_song = track_lookup[target_tid]

          negative_rows.append({
              "source_pid": row.source_pid,
              "source_region_genre_id": row.source_region_genre_id,
              "source_position": row.source_position,
              "source_pid_position": row.source_pid_position,
              "source_track_id": row.source_track_id,
              "source_Track Name": row.source_Track_Name,
              "source_Artist Name(s)": row.source_Artist_Name_s,
              "source_Danceability": row.source_Danceability,
              "source_Energy": row.source_Energy,
              "source_Key": row.source_Key,
              "source_Loudness": row.source_Loudness,
              "source_Mode": row.source_Mode,
              "source_Speechiness": row.source_Speechiness,
              "source_Acousticness": row.source_Acousticness,
              "source_Instrumentalness": row.source_Instrumentalness,
              "source_Liveness": row.source_Liveness,
              "source_Valence": row.source_Valence,
              "source_Tempo": row.source_Tempo,

              "target_track_id": target_tid,
              "target_Track Name": target_song["Track Name"],
              "target_Artist Name(s)": target_song["Artist Name(s)"],
              "target_Danceability": target_song["Danceability"],
              "target_Energy": target_song["Energy"],
              "target_Key": target_song["Key"],
              "target_Loudness": target_song["Loudness"],
              "target_Mode": target_song["Mode"],
              "target_Speechiness": target_song["Speechiness"],
              "target_Acousticness": target_song["Acousticness"],
              "target_Instrumentalness": target_song["Instrumentalness"],
              "target_Liveness": target_song["Liveness"],
              "target_Valence": target_song["Valence"],
              "target_Tempo": target_song["Tempo"],

              "label": 0
          })

  negative_pairs = pd.DataFrame(negative_rows)

  print("Negative pairs:", len(negative_pairs))
  negative_pairs.head()

  # CELL A4: Combine and save labeled pair dataset

  positive_export = positive_pairs.copy()

  labeled_pairs = pd.concat(
      [positive_export, negative_pairs],
      ignore_index=True
  ).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

  pairs_path = os.path.join(processed_dir, "phase1_labeled_pairs.csv")
  labeled_pairs.to_csv(pairs_path, index=False)

  print("Saved:", pairs_path)
  print(labeled_pairs["label"].value_counts())
  print(labeled_pairs[["label", "source_track_id", "target_track_id"]].head())

  # CELL B1: Build Phase 1 feature matrix
  # Uses absolute differences for continuous features
  # Uses circle-of-fifths distance for Key
  # Uses absolute difference for Mode

  continuous_features = [
      "Danceability", "Energy", "Loudness", "Speechiness",
      "Acousticness", "Instrumentalness", "Liveness",
      "Valence", "Tempo"
  ]

  circle_of_fifths_order = [0, 7, 2, 9, 4, 11, 6, 1, 8, 3, 10, 5]
  fifths_pos = {k: i for i, k in enumerate(circle_of_fifths_order)}

  def circle_of_fifths_distance(k1, k2):
      k1 = int(k1)
      k2 = int(k2)
      p1 = fifths_pos[k1]
      p2 = fifths_pos[k2]
      diff = abs(p1 - p2)
      return min(diff, 12 - diff) / 6.0  # normalized to [0, 1]

  phase1_df = labeled_pairs.copy()

  for feat in continuous_features:
      phase1_df[f"diff_{feat.lower()}"] = (
          phase1_df[f"source_{feat}"] - phase1_df[f"target_{feat}"]
      ).abs()

  phase1_df["diff_mode"] = (
      phase1_df["source_Mode"] - phase1_df["target_Mode"]
  ).abs()

  phase1_df["diff_key_fifths"] = phase1_df.apply(
      lambda r: circle_of_fifths_distance(r["source_Key"], r["target_Key"]),
      axis=1
  )

  phase1_feature_cols = [
      "diff_danceability",
      "diff_energy",
      "diff_loudness",
      "diff_speechiness",
      "diff_acousticness",
      "diff_instrumentalness",
      "diff_liveness",
      "diff_valence",
      "diff_tempo",
      "diff_mode",
      "diff_key_fifths"
  ]

  X = phase1_df[phase1_feature_cols]
  y = phase1_df["label"]

  feature_matrix_path = os.path.join(processed_dir, "phase1_feature_matrix.csv")
  phase1_df[phase1_feature_cols + ["label"]].to_csv(feature_matrix_path, index=False)

  print("Saved:", feature_matrix_path)
  print("Shape:", X.shape)
  X.head()

  # CELL B2: Train/validation split

  X_train, X_test, y_train, y_test = train_test_split(
      X, y,
      test_size=0.2,
      random_state=RANDOM_STATE,
      stratify=y
  )

  print("Train shape:", X_train.shape)
  print("Test shape:", X_test.shape)
  print("Train label distribution:")
  print(y_train.value_counts(normalize=True))

  # CELL B3: Train XGBoost Phase 1 model

  try:
      from xgboost import XGBClassifier
  except ImportError:
      raise ImportError(
          "xgboost is not installed. Run this in a notebook cell first:\n\n%pip install xgboost"
      )

  phase1_model = XGBClassifier(
      n_estimators=300,
      max_depth=5,
      learning_rate=0.05,
      subsample=0.9,
      colsample_bytree=0.9,
      objective="binary:logistic",
      eval_metric="auc",
      random_state=RANDOM_STATE
  )

  phase1_model.fit(X_train, y_train)

  y_proba = phase1_model.predict_proba(X_test)[:, 1]
  y_pred = (y_proba >= 0.5).astype(int)

  auc = roc_auc_score(y_test, y_proba)

  print("Phase 1 ROC-AUC:", round(auc, 4))
  print()
  print(classification_report(y_test, y_pred, digits=4))

  # CELL B4: Extract normalized feature weights

  raw_importance = pd.Series(
      phase1_model.feature_importances_,
      index=phase1_feature_cols
  ).sort_values(ascending=False)

  normalized_weights = raw_importance / raw_importance.sum()

  weights_df = normalized_weights.reset_index()
  weights_df.columns = ["feature", "normalized_weight"]

  weights_path = os.path.join(processed_dir, "phase1_feature_weights.csv")
  weights_df.to_csv(weights_path, index=False)

  print("Saved:", weights_path)
  display(weights_df)

  # CELL B5: Optional quick acceptance checks

  print("Acceptance checks")
  print("-" * 30)
  print("1. Positive pairs:", (labeled_pairs["label"] == 1).sum())
  print("2. Negative pairs:", (labeled_pairs["label"] == 0).sum())
  print("3. Total pairs:", len(labeled_pairs))
  print("4. Phase 1 AUC:", round(auc, 4))
  print("5. Top 5 weighted features:")
  display(weights_df.head(5))

  What I want you to send me after running this:

  - positive pair count
  - negative pair count
  - total pair count
  - Phase 1 ROC-AUC
  - the top 5 rows of weights_df

  If you get errors, the most likely ones are:

  1. xgboost is not installed
     Use:

  %pip install xgboost

  2. Sample larger than population
     This would happen only if a playlist exclusion leaves too few valid targets. Replace this line:

  sampled_targets = random.sample(valid_targets, k=min(NEGATIVE_RATIO, len(valid_targets)))

  with:

  if len(valid_targets) == 0:
      continue
  sampled_targets = random.sample(valid_targets, k=min(NEGATIVE_RATIO, len(valid_targets)))

  3. Column name issues from notebook display/export differences
     Run:

  print(df_clean.columns.tolist())

  and send me the output if any column names differ.

  And on your question directly: yes, discard that last old cell from the active workflow. It is not the right training-data implementation for this project.